# Bike Sharing Demand Prediction

**Dataset:** UCI Bike Sharing (Capital Bikeshare, Washington DC, 2011–2012) — 17,379 hourly records with weather, season, hour and day type. **Task:** predict how many bikes will be rented in an hour (`cnt`) — the number an operator needs for rebalancing vans and staffing.

In [1]:
import io, urllib.request, zipfile
import pandas as pd

def get(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=90) as r: return r.read()

zf = zipfile.ZipFile(io.BytesIO(get('https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip')))
name = [n for n in zf.namelist() if n.endswith('hour.csv')][0]
d = pd.read_csv(io.BytesIO(zf.read(name)))
print(d.shape)
d.head()

(17379, 17)


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# casual + registered sum to cnt — leaking them would be cheating, so they are excluded
X = pd.get_dummies(d.drop(columns=['instant', 'dteday', 'casual', 'registered', 'cnt']),
                   columns=['season', 'weathersit', 'mnth', 'hr', 'weekday'], drop_first=True)
y = d['cnt']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1).fit(Xtr, ytr)
pred = rf.predict(Xte)
print('R2:', round(r2_score(yte, pred), 4))
print('MAE (bikes/hour):', round(mean_absolute_error(yte, pred), 2))
print('Naive always-average MAE:', round(float(np.mean(np.abs(yte - ytr.mean()))), 2))
imp = sorted(zip(X.columns, rf.feature_importances_), key=lambda t: -t[1])[:6]
print('Top features:', imp)

R2: 0.918
MAE (bikes/hour): 33.16
Naive always-average MAE: 140.88
Top features: [('atemp', np.float64(0.1586215048668004)), ('workingday', np.float64(0.09490050965878821)), ('hr_17', np.float64(0.09314655285914121)), ('yr', np.float64(0.08062026328516268)), ('hum', np.float64(0.07811267273580055)), ('hr_18', np.float64(0.07203417763860638))]


## What I learned
- Hour of day dominates: commuting peaks at 8am and 5–6pm are the demand story.
- Leakage discipline matters: `casual` + `registered` literally sum to the target — including them would fake a near-perfect score.
- A random split is optimistic for time data (weather seasons repeat); a chronological split is the stricter follow-up.